# Custom Model

### Importing Libraries

In [1]:
import tensorflow as tf
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
from glob import glob
import matplotlib.pyplot as plt


In [2]:
tf.__version__

'2.14.0'

### Data Preprocessing

In [3]:
import numpy as np
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define a custom preprocessing function that combines imadjust, Wiener deconvolution, and histogram equalization
def custom_preprocessing(image):
    # 1: Applying imadjust (adjust intensity values)
    in_low = 50   
    in_high = 200 
    out_low = 0   
    out_high = 255   
    image = np.interp(image, (in_low, in_high), (out_low, out_high)).astype(np.uint8)

    # Step 2: Apply Wiener deconvolution for image restoration
    psf = np.ones((5, 5)) / 25
    image_restored = cv2.filter2D(image, -1, psf)

    # Step 3: Apply histogram equalization
    if image.shape[-1] == 3:  # Check if the image has 3 channels (RGB)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    
    #R, G, B = cv2.split(image)

    #output1_R = cv2.equalizeHist(R)
    #output1_G = cv2.equalizeHist(G)
    #output1_B = cv2.equalizeHist(B)

    #equalized_image = cv2.merge((output1_R, output1_G, output1_B))
    equalized_image = cv2.equalizeHist(image_restored)
    
    return equalized_image

    #image = cv2.imread('path_to_image.jpg', cv2.IMREAD_GRAYSCALE).astype(np.uint8)


In [4]:
# Creating an ImageDataGenerator with the custom preprocessing function
train_datagen = ImageDataGenerator(
    #preprocessing_function=custom_preprocessing,
    rescale = 1./255,
    shear_range=0.2,
    zoom_range=0.2
)

test_datagen = ImageDataGenerator(
    #preprocessing_function=custom_preprocessing,
    rescale = 1./255
)


In [5]:
training_set = train_datagen.flow_from_directory('Dataset/train',
                                                 target_size = (224, 224),
                                                 batch_size = 32,
                                                 class_mode = 'categorical')

Found 12121 images belonging to 3 classes.


In [6]:
test_set = test_datagen.flow_from_directory('Dataset/test',
                                            target_size = (224, 224),
                                            batch_size = 32,
                                            class_mode = 'categorical')

Found 3032 images belonging to 3 classes.


In [7]:
folders = glob('Dataset/train/*')

### CNN Model

In [11]:
from tensorflow.keras import layers, models

# Defining the CNN model architecture
cnn = models.Sequential()

# Adding layers

# layer 1 = Covn layer   7×7 Covn, 64, Stride 2 
cnn.add(layers.Conv2D(filters=64, kernel_size=7, strides=2,padding='valid', activation='relu',input_shape=[224,224,3]))

# layer 2 = MPL   3×3 max pool, 2S
cnn.add(layers.MaxPool2D(pool_size=3, strides=2))

# Residual Layer 1 :  1×1 Conv, 128 →ReLU ;3×3 Covn, 128 →ReLU ;1×1 Covn, 256 →ReLU

cnn.add(layers.Conv2D(filters=128,strides=2, kernel_size=1,activation='relu'))
cnn.add(layers.Conv2D(filters=128, kernel_size=3,activation='relu'))
cnn.add(layers.Conv2D(filters=256, kernel_size=1,activation='relu'))

# Residual Layer 2 :  1×1 Conv, 256 →ReLU ;3×3 Covn, 256 →ReLU ;1×1 Covn, 512 →ReLU

cnn.add(layers.Conv2D(filters=256, kernel_size=1,activation='relu'))
cnn.add(layers.Conv2D(filters=256, kernel_size=3,activation='relu'))
cnn.add(layers.Conv2D(filters=512, kernel_size=1,activation='relu'))

# Residual layer 3 :  1×1 Conv, 512→ReLU ;3×3 Covn, 512 →ReLU ;1×1 Covn, 1024 →ReLU

cnn.add(layers.Conv2D(filters=512, kernel_size=1,activation='relu'))
cnn.add(layers.Conv2D(filters=512, kernel_size=3,activation='relu'))
cnn.add(layers.Conv2D(filters=1024, kernel_size=1,activation='relu'))

# Residual layer 4:  1×1 Conv, 1024 →ReLU ;3×3 Covn, 1024 →ReLU ;1×1 Covn, 2048 →ReLU

cnn.add(layers.Conv2D(filters=1024, kernel_size=1,activation='relu'))
cnn.add(layers.Conv2D(filters=1024, kernel_size=3,activation='relu'))
cnn.add(layers.Conv2D(filters=2048, kernel_size=1,activation='relu'))

# Global Average pooling

cnn.add(layers.GlobalAveragePooling2D())

# Full Connection

cnn.add(layers.Dense(units=256, activation= 'relu'))

# output layer

cnn.add(layers.Dense(units=len(folders), activation= 'softmax'))


cnn.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_13 (Conv2D)          (None, 109, 109, 64)      9472      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 54, 54, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_14 (Conv2D)          (None, 27, 27, 128)       8320      
                                                                 
 conv2d_15 (Conv2D)          (None, 25, 25, 128)       147584    
                                                                 
 conv2d_16 (Conv2D)          (None, 25, 25, 256)       33024     
                                                                 
 conv2d_17 (Conv2D)          (None, 25, 25, 256)       65792     
                                                      

### Training the CNN

In [12]:
# Compiling the CNN

cnn.compile(
  loss='categorical_crossentropy',
  optimizer='adam',
  metrics=['accuracy']
)

In [13]:
# Training the CNN on the training set and evaluating it on the Test set

r = cnn.fit_generator(
  training_set,
  validation_data=test_set,
  epochs=25,
  steps_per_epoch=len(training_set),
  validation_steps=len(test_set)
)

C:\Users\rishe\AppData\Local\Temp\ipykernel_26520\251603458.py:3: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  r = cnn.fit_generator(


Epoch 1/25
  7/379 [..............................] - ETA: 3:11:48 - loss: 1.1536 - accuracy: 0.5893

KeyboardInterrupt: 

In [ ]:
# plot the loss
plt.plot(r.history['loss'], label='train loss')
plt.plot(r.history['val_loss'], label='val loss')
plt.legend()
plt.show()
plt.savefig('LossVal_loss')

# plot the accuracy
plt.plot(r.history['acc'], label='train acc')
plt.plot(r.history['val_acc'], label='val acc')
plt.legend()
plt.show()
plt.savefig('AccVal_acc')